# analisis_segunda.ipynb

Pipeline completo de transformación y análisis de LaLiga 2 (Segunda División) 25/26.

## Orden de ejecución
1. `data_segunda_advanced.py` → `segunda_division_stats.json` + `segunda_division_stats.parquet`
2. **Este notebook** → `dashboard_data.json` + parquets Power BI
3. `update_dashboard.py` → inyecta el JSON en `dashboard.html`

## Celdas
- **Celda 1** — Carga y validación del parquet
- **Celda 2** — Métricas derivadas (todas las columnas calculadas)
- **Celda 3** — Tabla de clasificación
- **Celda 4** — Análisis Córdoba + generación de `dashboard_data.json`
- **Celda 5** — Exportación parquets Power BI (star schema)
- **Celda 6** — Diagnóstico y verificación final

In [7]:
# CELDA 1 — CARGA Y VALIDACIÓN DEL PARQUET

import pandas as pd
import numpy as np
import json

PARQUET_FILE = "../data/processed/segunda_division_stats.parquet"
CORDOBA_ID   = 2850
OUTPUT_JSON  = "../data/processed/dashboard_data.json"

df = pd.read_parquet(PARQUET_FILE)

# Validación de estructura mínima
REQUIRED_COLS = [
    "match_id", "round", "team_id", "team_name", "is_home",
    "goals_for", "goals_against", "start_timestamp",
    "expectedGoals_ALL", "ballPossession_ALL", "totalShotsOnGoal_ALL",
    "shotsOnGoal_ALL", "totalTackle_ALL", "interceptionWon_ALL",
    "ballRecovery_ALL", "touchesInOppBox_ALL", "duelWonPercent_ALL",
    "bigChanceCreated_ALL", "passes_ALL", "accuratePasses_ALL",
]
missing = [c for c in REQUIRED_COLS if c not in df.columns]
if missing:
    raise ValueError(
        f"El parquet no tiene la estructura esperada. Columnas faltantes:\n{missing}\n"
        "Solución: vuelve a ejecutar data_segunda_advanced.py para regenerar el parquet."
    )

# Detección de columnas opcionales
HAS_1ST    = "ballPossession_1ST" in df.columns
HAS_2ND    = "ballPossession_2ND" in df.columns
HAS_TOTALS = "accurateLongBalls_total_ALL" in df.columns

# Stats nuevas incorporadas en esta versión del notebook
NEW_STATS = [
    "totalShotsInsideBox_ALL", "totalShotsOutsideBox_ALL", "totalClearance_ALL",
    "errorsLeadToShot_ALL", "dispossessed_ALL", "offsides_ALL",
    "freeKicks_ALL", "fouledFinalThird_ALL", "accurateThroughBall_ALL",
    "hitWoodwork_ALL", "highClaims_ALL", "goalKicks_ALL",
    "bigChanceScored_ALL", "bigChanceMissed_ALL",
    "cornerKicks_ALL", "fouls_ALL", "finalThirdEntries_ALL",
    "goalkeeperSaves_ALL",
]
new_present = [c for c in NEW_STATS if c in df.columns]
new_absent  = [c for c in NEW_STATS if c not in df.columns]

# Detección automática del Córdoba
cordoba_names = [n for n in df["team_name"].unique() if "rdoba" in n or "ordoba" in n]
if not cordoba_names:
    raise ValueError("No se encontró el Córdoba en el parquet. Revisar columna team_name.")
CORDOBA_NAME = cordoba_names[0]

print("✓ Parquet cargado correctamente")
print(f"  Filas:             {len(df)}")
print(f"  Columnas:          {len(df.columns)}")
print(f"  Equipos:           {df['team_name'].nunique()}")
print(f"  Partidos:          {df['match_id'].nunique()}")
print(f"  Jornadas:          {sorted(df['round'].dropna().unique().astype(int).tolist())}")
print(f"  Córdoba:           {CORDOBA_ID} → '{CORDOBA_NAME}'")
print(f"  Períodos 1ST/2ND:  {HAS_1ST} / {HAS_2ND}")
print(f"  Columnas _total_*: {HAS_TOTALS}")
print(f"  Stats nuevas OK:   {len(new_present)}/{len(NEW_STATS)}")
if new_absent:
    print(f"  Stats AUSENTES:    {new_absent}")
    print("  → Regenerar parquet con data_segunda_advanced.py")
print()
print("Equipos:", sorted(df["team_name"].unique()))

✓ Parquet cargado correctamente
  Filas:             660
  Columnas:          154
  Equipos:           22
  Partidos:          330
  Jornadas:          [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30]
  Córdoba:           2850 → 'Córdoba'
  Períodos 1ST/2ND:  True / True
  Columnas _total_*: True
  Stats nuevas OK:   18/18

Equipos: ['AD Ceuta', 'Albacete Balompié', 'Almería', 'Burgos Club de Fútbol', 'CD Castellón', 'Cultural Leonesa', 'Cádiz', 'Córdoba', 'Deportivo La Coruña', 'Eibar', 'FC Andorra', 'Granada', 'Huesca', 'Las Palmas', 'Leganés', 'Mirandés', 'Málaga', 'Real Racing Club', 'Real Sociedad B U21', 'Real Valladolid', 'Real Zaragoza', 'Sporting Gijón']


In [8]:
# CELDA 2 — MÉTRICAS DERIVADAS
#
# STATS NUEVAS INCORPORADAS
# --------------------------
# Alta relevancia:
#   shots_in_box / shots_out_box  → calidad posicional del ataque
#   shots_in_box_pct              → % tiros dentro del área (métrica derivada)
#   clearance                     → despejes (presión defensiva recibida)
#   errors_lead_shot              → errores que generan tiro rival
#   dispossessed                  → pérdidas de balón en duelo
#   offsides                      → señal de línea alta y transiciones
#
# Media relevancia:
#   free_kicks / foul_balance     → faltas recibidas y balance táctico
#   fouled_final_third            → pressing ofensivo
#   through_balls                 → juego vertical
#   hit_woodwork                  → oportunidades de gol / mala suerte
#   high_claims / goal_kicks      → actividad del portero y presión rival
#   bigC_scored / bigC_missed     → eficiencia en grandes ocasiones
#   corners / fouls               → set pieces y disciplina
#   final_third_entries           → penetración ofensiva
#   gk_saves                      → trabajo del portero

# ── Helpers ──────────────────────────────────────────────────────────────────
def safe_ratio(dataframe, num_col, den_col):
    """Ratio seguro: NaN si columna ausente o denominador 0."""
    if num_col in dataframe.columns and den_col in dataframe.columns:
        return dataframe[num_col] / dataframe[den_col].replace(0, np.nan)
    return np.nan

def safe_col(dataframe, col):
    """Convierte a numérico si existe, NaN si no existe."""
    if col in dataframe.columns:
        return pd.to_numeric(dataframe[col], errors="coerce")
    return np.nan

# ─Resultado y puntos 
df["result"] = np.where(
    df["goals_for"] > df["goals_against"], "W",
    np.where(df["goals_for"] == df["goals_against"], "D", "L")
)
df["points"]    = df["result"].map({"W": 3, "D": 1, "L": 0})
df["goal_diff"] = df["goals_for"] - df["goals_against"]

# TIROS 
df["total_shots"]   = pd.to_numeric(df["totalShotsOnGoal_ALL"], errors="coerce")
df["shotsOT"]       = pd.to_numeric(df["shotsOnGoal_ALL"],      errors="coerce")
df["shots_in_box"]  = safe_col(df, "totalShotsInsideBox_ALL")   # NUEVO
df["shots_out_box"] = safe_col(df, "totalShotsOutsideBox_ALL")  # NUEVO
df["hit_woodwork"]  = safe_col(df, "hitWoodwork_ALL")           # NUEVO
df["shots_on_pct"]     = safe_ratio(df, "shotsOnGoal_ALL", "totalShotsOnGoal_ALL")
df["shots_in_box_pct"] = safe_ratio(df, "totalShotsInsideBox_ALL", "totalShotsOnGoal_ALL")  # NUEVO

# xG Y CONVERSIÓN 
df["xg"]         = pd.to_numeric(df["expectedGoals_ALL"], errors="coerce")
df["conversion"] = df["goals_for"] / df["xg"].replace(0, np.nan)
df["xg_over"]    = df["goals_for"] - df["xg"]

# xGA (xG concedido = xG del rival en el mismo partido) 
_xg_pairs   = df[["match_id", "team_id", "xg"]].copy()
_xga_lookup = (
    _xg_pairs
    .merge(_xg_pairs, on="match_id", suffixes=("", "_opp"))
    .query("team_id != team_id_opp")
    [["match_id", "team_id", "xg_opp"]]
    .rename(columns={"xg_opp": "xga"})
)
df = df.merge(_xga_lookup, on=["match_id", "team_id"], how="left")
df.loc[df["xga"] == 0.0, "xga"] = np.nan
df["xg_net"] = df["xg"] - df["xga"]  # dominio real del partido

# GRANDES OCASIONES 
df["bigC"]          = pd.to_numeric(df["bigChanceCreated_ALL"], errors="coerce")
df["bigC_scored"]   = safe_col(df, "bigChanceScored_ALL")   # NUEVO alias explícito
df["bigC_missed"]   = safe_col(df, "bigChanceMissed_ALL")   # NUEVO
df["big_chance_eff"]  = safe_ratio(df, "bigChanceScored_ALL", "bigChanceCreated_ALL")
df["big_chance_conv"] = df["big_chance_eff"]  # alias más descriptivo

# POSESIÓN Y PASES
df["poss"]                = pd.to_numeric(df["ballPossession_ALL"], errors="coerce")
df["passes_col"]          = pd.to_numeric(df["passes_ALL"],         errors="coerce")
df["pass_accuracy"]       = safe_ratio(df, "accuratePasses_ALL", "passes_ALL")
df["long_ball_accuracy"]  = safe_ratio(df, "accurateLongBalls_ALL", "accurateLongBalls_total_ALL")
df["cross_accuracy"]      = safe_ratio(df, "accurateCross_ALL",     "accurateCross_total_ALL")
df["final_third_pass_pct"]= safe_ratio(df, "finalThirdPhaseStatistic_ALL", "finalThirdPhaseStatistic_total_ALL")
df["final_third_entries"] = safe_col(df, "finalThirdEntries_ALL")  # NUEVO alias
df["through_balls"]       = safe_col(df, "accurateThroughBall_ALL") # NUEVO

# ÁREA RIVAL
df["touches"]       = pd.to_numeric(df["touchesInOppBox_ALL"], errors="coerce")
df["box_dominance"] = safe_ratio(df, "touchesInOppBox_ALL", "totalShotsOnGoal_ALL")

# DISCIPLINA Y FALTAS
df["fouls"]              = safe_col(df, "fouls_ALL")           # NUEVO alias
df["free_kicks"]         = safe_col(df, "freeKicks_ALL")       # NUEVO
df["corners"]            = safe_col(df, "cornerKicks_ALL")     # NUEVO
df["offsides"]           = safe_col(df, "offsides_ALL")        # NUEVO
df["fouled_final_third"] = safe_col(df, "fouledFinalThird_ALL") # NUEVO
# Balance de faltas: positivo = más faltas recibidas que cometidas
if "freeKicks_ALL" in df.columns and "fouls_ALL" in df.columns:
    df["foul_balance"] = safe_col(df, "freeKicks_ALL") - safe_col(df, "fouls_ALL")  # NUEVO

# DEFENSA
df["tackle"]           = pd.to_numeric(df["totalTackle_ALL"],    errors="coerce")
df["inter"]            = pd.to_numeric(df["interceptionWon_ALL"], errors="coerce")
df["recov"]            = pd.to_numeric(df["ballRecovery_ALL"],   errors="coerce")
df["clearance"]        = safe_col(df, "totalClearance_ALL")      # NUEVO
df["errors_lead_shot"] = safe_col(df, "errorsLeadToShot_ALL")    # NUEVO
df["dispossessed"]     = safe_col(df, "dispossessed_ALL")         # NUEVO
df["tackle_won_pct"]   = safe_ratio(df, "wonTacklePercent_ALL", "wonTacklePercent_total_ALL")
df["ground_duel_pct"]  = safe_ratio(df, "groundDuelsPercentage_ALL", "groundDuelsPercentage_total_ALL")
df["aerial_duel_pct"]  = safe_ratio(df, "aerialDuelsPercentage_ALL", "aerialDuelsPercentage_total_ALL")
df["dribble_pct"]      = safe_ratio(df, "dribblesPercentage_ALL",   "dribblesPercentage_total_ALL")
df["duels"]            = pd.to_numeric(df["duelWonPercent_ALL"], errors="coerce")
df["pressure_index"]   = df["tackle"] + df["inter"] + df["recov"]

# PORTERÍA
df["gk_saves"]    = safe_col(df, "goalkeeperSaves_ALL")  # NUEVO alias
df["high_claims"] = safe_col(df, "highClaims_ALL")        # NUEVO
df["goal_kicks"]  = safe_col(df, "goalKicks_ALL")         # NUEVO

# DIFERENCIALES POR MITAD
if HAS_1ST and HAS_2ND:
    df["possession_shift"] = (
        pd.to_numeric(df["ballPossession_2ND"],   errors="coerce") -
        pd.to_numeric(df["ballPossession_1ST"],   errors="coerce")
    )
    df["shots_shift"] = (
        pd.to_numeric(df["totalShotsOnGoal_2ND"], errors="coerce") -
        pd.to_numeric(df["totalShotsOnGoal_1ST"], errors="coerce")
    )
    if "expectedGoals_2ND" in df.columns and "expectedGoals_1ST" in df.columns:
        df["xg_shift"] = (
            pd.to_numeric(df["expectedGoals_2ND"], errors="coerce") -
            pd.to_numeric(df["expectedGoals_1ST"], errors="coerce")
        )
    print("✓ Diferenciales por mitad calculados")
else:
    print("⚠ Sin datos por mitad — possession_shift / shots_shift no calculados")

# ACUMULADOS POR JORNADA
if "round" in df.columns:
    df = df.sort_values(["team_id", "round"]).copy()
    df["cumulative_points"] = df.groupby("team_id")["points"].cumsum()
    df["cumulative_gd"]     = df.groupby("team_id")["goal_diff"].cumsum()
    df["form_last5"]        = (
        df.groupby("team_id")["points"]
        .transform(lambda x: x.rolling(5, min_periods=1).mean())
    )
    print("✓ Acumulados por jornada calculados")

# FECHA DEL PARTIDO
if "start_timestamp" in df.columns:
    df["match_date"] = pd.to_datetime(df["start_timestamp"], unit="s", utc=True)
    print("✓ match_date generado")

print("\n✓ Celda 2 completada — métricas derivadas calculadas")

✓ Diferenciales por mitad calculados
✓ Acumulados por jornada calculados
✓ match_date generado

✓ Celda 2 completada — métricas derivadas calculadas


In [9]:
# CELDA 3 — TABLA DE CLASIFICACIÓN

table = df.groupby("team_id").agg(
    team_name      = ("team_name",      "first"),
    matches        = ("points",         "count"),
    points         = ("points",         "sum"),
    wins           = ("result",         lambda x: (x == "W").sum()),
    draws          = ("result",         lambda x: (x == "D").sum()),
    losses         = ("result",         lambda x: (x == "L").sum()),
    goals_for      = ("goals_for",      "sum"),
    goals_against  = ("goals_against",  "sum"),
    goal_diff      = ("goal_diff",      "sum"),
    xg_sum         = ("xg",             "sum"),
    xg_count       = ("xg",             "count"),
    xga_mean       = ("xga",            "mean"),
    avg_possession = ("poss",           "mean"),
    avg_pressure   = ("pressure_index", "mean"),
    avg_pass_acc   = ("pass_accuracy",  "mean"),
    avg_shots      = ("total_shots",    "mean"),
    avg_shotsOT    = ("shotsOT",        "mean"),
    avg_bigC       = ("bigC",           "mean"),
).reset_index()

table["ppg"]         = table["points"] / table["matches"]
table["xg_per_game"] = table["xg_sum"] / table["xg_count"]

# conv: goles / xG acumulado (solo partidos con xG disponible)
xg_mask      = df["xg"].notna()
goles_con_xg = df[xg_mask].groupby("team_id")["goals_for"].sum().rename("goals_with_xg")
table        = table.merge(goles_con_xg, on="team_id", how="left")
table["conv"] = table["goals_with_xg"] / table["xg_sum"].replace(0, np.nan)

# Desempate oficial LaLiga: pts → goal_diff → goals_for
table = table.sort_values(
    ["points", "goal_diff", "goals_for"],
    ascending=[False, False, False]
).reset_index(drop=True)
table["position"] = table.index + 1

table["group"] = pd.cut(
    table["position"],
    bins=[0, 2, 6, 19, 22],
    labels=["Ascenso directo", "Playoff ascenso", "Zona media", "Descenso"],
    ordered=False
)

top6_ids = table.head(6)["team_id"].tolist()

print("\n=== CLASIFICACIÓN J" + str(int(df['round'].max())) + " ===")
print(table[["position", "team_name", "matches", "points", "wins", "draws",
             "losses", "goals_for", "goals_against", "goal_diff",
             "ppg", "xg_per_game", "xga_mean", "conv", "group"]].to_string(index=False))


=== CLASIFICACIÓN J30 ===
 position             team_name  matches  points  wins  draws  losses  goals_for  goals_against  goal_diff      ppg  xg_per_game  xga_mean     conv           group
        1      Real Racing Club       30      59    18      5       7         62             40         22 1.966667     1.641034  1.402759 1.281782 Ascenso directo
        2               Almería       30      52    15      7       8         56             42         14 1.733333     1.534828  1.500000 1.213211 Ascenso directo
        3   Deportivo La Coruña       30      52    15      7       8         46             34         12 1.733333     1.435333  1.257667 1.068277 Playoff ascenso
        4                Málaga       30      51    15      6       9         49             37         12 1.700000     1.472667  1.403333 1.109099 Playoff ascenso
        5          CD Castellón       30      49    14      7       9         49             37         12 1.633333     1.811000  1.202000 0.901896 Playo

In [10]:
# CELDA 4 — ANÁLISIS CÓRDOBA + GENERACIÓN DE dashboard_data.json

cordoba = df[df["team_id"] == CORDOBA_ID].copy().sort_values("round").reset_index(drop=True)
if cordoba.empty:
    raise ValueError(f"No se encontró el Córdoba con team_id={CORDOBA_ID}.")

# Nombre del rival 
rival_map = (
    df[df["team_id"] != CORDOBA_ID]
    .groupby("match_id")["team_name"]
    .first()
    .to_dict()
)
cordoba["rival"] = cordoba["match_id"].map(rival_map).fillna("Desconocido")

# Helper para exportar valores al JSON 
def jval(row, col, decimals=1):
    """Valor redondeado o None si NaN / columna ausente."""
    if col not in row.index or pd.isna(row[col]):
        return None
    v = float(row[col])
    return round(v, decimals) if decimals > 0 else int(v)

# PARTIDOS para el dashboard 
partidos_js = []
for _, row in cordoba.iterrows():
    partidos_js.append({
        # Base
        "j":             int(row["round"]),
        "local":         "L" if row["is_home"] else "V",
        "rival":         row["rival"],
        "gf":            int(row["goals_for"]),
        "ga":            int(row["goals_against"]),
        "res":           row["result"],
        # xG
        "xg":            jval(row, "xg", 2),
        "xga":           jval(row, "xga", 2),
        # Tiros
        "shots":         jval(row, "total_shots"),
        "shotsOT":       jval(row, "shotsOT"),
        "shotsInBox":    jval(row, "shots_in_box"),    # NUEVO
        "shotsOutBox":   jval(row, "shots_out_box"),   # NUEVO
        "hitWoodwork":   jval(row, "hit_woodwork"),    # NUEVO
        # Grandes ocasiones
        "bigC":          jval(row, "bigC"),
        "bigCScored":    jval(row, "bigC_scored"),     # NUEVO
        "bigCMissed":    jval(row, "bigC_missed"),     # NUEVO
        # Posesión y pases
        "poss":          jval(row, "poss"),
        "passes":        jval(row, "passes_col", 0),
        "finalThirdEnt": jval(row, "final_third_entries"),  # NUEVO
        "throughBalls":  jval(row, "through_balls"),   # NUEVO
        # Área rival
        "touches":       jval(row, "touches"),
        # Defensa
        "tackle":        jval(row, "tackle"),
        "inter":         jval(row, "inter"),
        "recov":         jval(row, "recov"),
        "clearance":     jval(row, "clearance"),       # NUEVO
        "errorsShot":    jval(row, "errors_lead_shot"), # NUEVO
        "dispossessed":  jval(row, "dispossessed"),    # NUEVO
        # Duelos
        "duels":         jval(row, "duels"),
        # Set pieces y disciplina
        "corners":       jval(row, "corners"),         # NUEVO
        "fouls":         jval(row, "fouls"),            # NUEVO
        "freeKicks":     jval(row, "free_kicks"),      # NUEVO
        "offsides":      jval(row, "offsides"),         # NUEVO
        "fouledFinal":   jval(row, "fouled_final_third"), # NUEVO
        # Portería
        "gkSaves":       jval(row, "gk_saves"),        # NUEVO
        "highClaims":    jval(row, "high_claims"),     # NUEVO
        "goalKicks":     jval(row, "goal_kicks"),      # NUEVO
    })

# CLASIFICACIÓN
liga_js = []
for _, row in table.iterrows():
    liga_js.append({
        "pos":  int(row["position"]),
        "name": row["team_name"],
        "pj":   int(row["matches"]),
        "pts":  int(row["points"]),
        "gf":   int(row["goals_for"]),
        "gc":   int(row["goals_against"]),
        "ppg":  round(float(row["ppg"]), 3),
        "xg":   round(float(row["xg_per_game"]), 2) if pd.notna(row["xg_per_game"]) else 0,
        "xga":  round(float(row["xga_mean"]), 2)    if pd.notna(row["xga_mean"])    else 0,
        "conv": round(float(row["conv"]), 2)         if pd.notna(row["conv"])        else 0,
    })

# MEDIAS DE LIGA
def liga_mean(col, decimals=2):
    if col in df.columns and df[col].notna().any():
        return round(float(df[col].mean()), decimals)
    return None

liga_medias = {
    "xg":            liga_mean("xg"),
    "xga":           liga_mean("xga"),
    "poss":          liga_mean("poss", 1),
    "shots":         liga_mean("total_shots"),
    "shotsOT":       liga_mean("shotsOT"),
    "shotsInBox":    liga_mean("shots_in_box"),
    "shotsOutBox":   liga_mean("shots_out_box"),
    "hitWoodwork":   liga_mean("hit_woodwork"),
    "bigC":          liga_mean("bigC"),
    "bigCScored":    liga_mean("bigC_scored"),
    "bigCMissed":    liga_mean("bigC_missed"),
    "touches":       liga_mean("touches"),
    "finalThirdEnt": liga_mean("final_third_entries"),
    "throughBalls":  liga_mean("through_balls"),
    "tackle":        liga_mean("tackle"),
    "inter":         liga_mean("inter"),
    "recov":         liga_mean("recov"),
    "clearance":     liga_mean("clearance"),
    "errorsShot":    liga_mean("errors_lead_shot"),
    "dispossessed":  liga_mean("dispossessed"),
    "duels":         liga_mean("duels"),
    "corners":       liga_mean("corners"),
    "fouls":         liga_mean("fouls"),
    "freeKicks":     liga_mean("free_kicks"),
    "offsides":      liga_mean("offsides"),
    "fouledFinal":   liga_mean("fouled_final_third"),
    "gkSaves":       liga_mean("gk_saves"),
    "highClaims":    liga_mean("high_claims"),
    "goalKicks":     liga_mean("goal_kicks"),
}

# BLOQUE CÓRDOBA
n_partidos = len(cordoba)
victorias  = int((cordoba["result"] == "W").sum())
empates    = int((cordoba["result"] == "D").sum())
derrotas   = int((cordoba["result"] == "L").sum())
pts_total  = int(cordoba["points"].sum())

cor_xg_valid = cordoba[cordoba["xg"].notna()]
conv_xg = (
    round(float(cor_xg_valid["goals_for"].sum() / cor_xg_valid["xg"].sum()), 2)
    if cor_xg_valid["xg"].sum() > 0 else None
)

def ppg_tramo(rows):
    return round(float(rows["points"].sum() / len(rows)), 3) if len(rows) > 0 else 0.0

def cor_mean(col, decimals=2):
    if col in cordoba.columns and cordoba[col].notna().any():
        return round(float(cordoba[col].mean()), decimals)
    return None

last5 = cordoba.tail(5)
last10= cordoba.tail(10)
loc   = cordoba[cordoba["is_home"] == True]
vis   = cordoba[cordoba["is_home"] == False]

cor_medias = {
    # Métricas de juego
    "xg":            cor_mean("xg"),
    "xga":           cor_mean("xga"),
    "poss":          cor_mean("poss", 1),
    "shots":         cor_mean("total_shots"),
    "shotsOT":       cor_mean("shotsOT"),
    "shotsInBox":    cor_mean("shots_in_box"),
    "shotsOutBox":   cor_mean("shots_out_box"),
    "hitWoodwork":   cor_mean("hit_woodwork"),
    "bigC":          cor_mean("bigC"),
    "bigCScored":    cor_mean("bigC_scored"),
    "bigCMissed":    cor_mean("bigC_missed"),
    "touches":       cor_mean("touches"),
    "finalThirdEnt": cor_mean("final_third_entries"),
    "throughBalls":  cor_mean("through_balls"),
    "tackle":        cor_mean("tackle"),
    "inter":         cor_mean("inter"),
    "recov":         cor_mean("recov"),
    "clearance":     cor_mean("clearance"),
    "errorsShot":    cor_mean("errors_lead_shot"),
    "dispossessed":  cor_mean("dispossessed"),
    "duels":         cor_mean("duels"),
    "corners":       cor_mean("corners"),
    "fouls":         cor_mean("fouls"),
    "freeKicks":     cor_mean("free_kicks"),
    "offsides":      cor_mean("offsides"),
    "fouledFinal":   cor_mean("fouled_final_third"),
    "gkSaves":       cor_mean("gk_saves"),
    "highClaims":    cor_mean("high_claims"),
    "goalKicks":     cor_mean("goal_kicks"),
    # Resultados
    "pts_total":  pts_total,
    "ppg":        round(float(pts_total / n_partidos), 3),
    "victorias":  victorias,
    "empates":    empates,
    "derrotas":   derrotas,
    "gf":         int(cordoba["goals_for"].sum()),
    "gc":         int(cordoba["goals_against"].sum()),
    "conv_xg":    conv_xg,
    "presion_idx":round(float(cordoba["pressure_index"].mean()), 1),
    "partidos":   n_partidos,
    # PPG por tramos (Monte Carlo)
    "ppg_last5":  ppg_tramo(last5),
    "ppg_last10": ppg_tramo(last10),
    # Separación local/visitante (Monte Carlo)
    "local_v":   int((loc["result"] == "W").sum()),
    "local_e":   int((loc["result"] == "D").sum()),
    "local_d":   int((loc["result"] == "L").sum()),
    "visit_v":   int((vis["result"] == "W").sum()),
    "visit_e":   int((vis["result"] == "D").sum()),
    "visit_d":   int((vis["result"] == "L").sum()),
    "local_ppg": ppg_tramo(loc),
    "visit_ppg": ppg_tramo(vis),
}

# CORRELACIONES con puntos (toda la liga)
corr_cols = {
    "shotsOT":           "Tiros puerta",
    "xg":                "xG",
    "xga":               "xGA (concedido)",
    "xg_net":            "xG neto (xG-xGA)",
    "bigC":              "Grandes oc.",
    "total_shots":       "Tiros tot.",
    "shots_in_box":      "Tiros en área",
    "touches":           "Toques área",
    "duels":             "Duelos gan.",
    "poss":              "Posesión",
    "tackle":            "Tackles",
    "recov":             "Recuperaciones",
    "inter":             "Interceptaciones",
    "clearance":         "Despejes",
    "final_third_entries": "Ent. último tercio",
    "through_balls":     "Pases profundidad",
    "fouls":             "Faltas cometidas",
    "free_kicks":        "Faltas recibidas",
    "corners":           "Córners",
    "offsides":          "Fueras de juego",
    "errors_lead_shot":  "Errores→tiro",
    "dispossessed":      "Pérdidas duelo",
    "big_chance_conv":   "Conv. grandes oc.",
    "aerial_duel_pct":   "Duelos aéreos %",
    "pass_accuracy":     "Precisión pases",
    "long_ball_accuracy":"Precisión pases largos",
    "tackle_won_pct":    "% tackles ganados",
    "shots_in_box_pct":  "% tiros en área",
    "fouled_final_third":"Faltas recib. último tercio",
}
correlaciones_js = []
for col_name, label in corr_cols.items():
    try:
        if col_name in df.columns and df[col_name].notna().sum() > 10:
            r = float(df[col_name].corr(df["points"]))
            if pd.notna(r):
                correlaciones_js.append({"label": label, "r": round(r, 3)})
    except Exception:
        pass
correlaciones_js.sort(key=lambda x: x["r"], reverse=True)

# ANÁLISIS COMPARATIVO ────────────────────────────────────────────────────
def league_conv():
    mask = df["xg"].notna()
    xg_s = df[mask]["xg"].sum()
    return round(float(df[mask]["goals_for"].sum() / xg_s), 3) if xg_s > 0 else None

def top6_conv():
    mask = df["xg"].notna() & df["team_id"].isin(top6_ids)
    xg_s = df[mask]["xg"].sum()
    return round(float(df[mask]["goals_for"].sum() / xg_s), 3) if xg_s > 0 else None

summary = {
    "ppg":                   round(float(cordoba["points"].mean()), 3),
    "league_ppg":            round(float(table["ppg"].mean()), 3),
    "top6_ppg":              round(float(table.head(6)["ppg"].mean()), 3),
    "avg_possession":        round(float(cordoba["poss"].mean()), 2),
    "league_possession":     round(float(df["poss"].mean()), 2),
    "top6_possession":       round(float(df[df["team_id"].isin(top6_ids)]["poss"].mean()), 2),
    "avg_xg":                round(float(cordoba["xg"].mean()), 3),
    "league_xg":             round(float(df["xg"].mean()), 3),
    "top6_xg":               round(float(df[df["team_id"].isin(top6_ids)]["xg"].mean()), 3),
    "avg_xga":               round(float(cordoba["xga"].mean()), 3) if cordoba["xga"].notna().any() else None,
    "league_xga":            round(float(df["xga"].mean()), 3) if df["xga"].notna().any() else None,
    "conversion":            round(conv_xg, 3) if conv_xg else None,
    "league_conversion":     league_conv(),
    "top6_conversion":       top6_conv(),
    "pressure":              round(float(cordoba["pressure_index"].mean()), 2),
    "league_pressure":       round(float(df["pressure_index"].mean()), 2),
    "top6_pressure":         round(float(df[df["team_id"].isin(top6_ids)]["pressure_index"].mean()), 2),
    "pass_accuracy":         round(float(cordoba["pass_accuracy"].mean()), 3),
    "league_pass_accuracy":  round(float(df["pass_accuracy"].mean()), 3),
}
for met, col in [
    ("aerial_duel_pct",    "aerial_duel_pct"),
    ("shots_in_box_pct",   "shots_in_box_pct"),
    ("big_chance_conv",    "big_chance_conv"),
    ("long_ball_accuracy", "long_ball_accuracy"),
    ("cross_accuracy",     "cross_accuracy"),
    ("tackle_won_pct",     "tackle_won_pct"),
]:
    if col in df.columns and df[col].notna().any():
        summary[met]              = round(float(cordoba[col].mean()), 3)
        summary[f"league_{met}"] = round(float(df[col].mean()), 3)

# Rendimiento por tramo de posesión
cordoba["possession_bin"] = pd.cut(
    cordoba["poss"],
    bins=[0, 40, 50, 60, 100],
    labels=["<40%", "40-50%", "50-60%", ">60%"],
    include_lowest=True
)
perf_by_poss = cordoba.groupby("possession_bin", observed=True).agg(
    matches    = ("points",    "count"),
    avg_points = ("points",    "mean"),
    avg_goals  = ("goals_for", "mean"),
    avg_xg     = ("xg",        "mean"),
).reset_index()

# Escribir dashboard_data.json
output = {
    "PARTIDOS":      partidos_js,
    "LIGA":          liga_js,
    "LIGA_MEDIAS":   liga_medias,
    "CORDOBA":       cor_medias,
    "CORRELACIONES": correlaciones_js,
}
with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

cor_pos = int(table[table["team_id"] == CORDOBA_ID]["position"].values[0])
print("=" * 55)
print(f"✓ Partidos del Córdoba:  {n_partidos}")
print(f"✓ Equipos en liga:       {len(liga_js)}")
print(f"✓ Posición:              {cor_pos}º")
print(f"✓ Puntos:                {pts_total} ({victorias}V {empates}E {derrotas}D)")
print(f"✓ PPG global:            {cor_medias['ppg']}")
print(f"✓ PPG last 10:           {cor_medias['ppg_last10']}")
print(f"✓ PPG last 5:            {cor_medias['ppg_last5']}")
print(f"✓ PPG local:             {cor_medias['local_ppg']}")
print(f"✓ PPG visitante:         {cor_medias['visit_ppg']}")
print(f"✓ xG/p:                  {cor_medias['xg']}  | Liga: {liga_medias['xg']}")
print(f"✓ xGA/p:                 {cor_medias['xga']}  | Liga: {liga_medias['xga']}")
print(f"✓ conv_xg:               {cor_medias['conv_xg']}")
print(f"✓ Campos por partido:    {len(partidos_js[0])}")
print(f"✓ {OUTPUT_JSON} generado")
print("=" * 55)
print("\n=== COMPARACIÓN CÓRDOBA vs LIGA ===")
for k, v in summary.items():
    print(f"  {k:35s} {round(v, 3) if isinstance(v, float) else v}")
print("\n=== TOP 10 CORRELACIONES CON PUNTOS ===")
for c in correlaciones_js[:10]:
    print(f"  {c['label']:35s} r={c['r']}")
print("\n=== RENDIMIENTO SEGÚN POSESIÓN ===")
print(perf_by_poss.to_string(index=False))

✓ Partidos del Córdoba:  30
✓ Equipos en liga:       22
✓ Posición:              11º
✓ Puntos:                41 (11V 8E 11D)
✓ PPG global:            1.367
✓ PPG last 10:           1.2
✓ PPG last 5:            0.6
✓ PPG local:             1.4
✓ PPG visitante:         1.333
✓ xG/p:                  1.66  | Liga: 1.35
✓ xGA/p:                 1.42  | Liga: 1.35
✓ conv_xg:               0.8
✓ Campos por partido:    36
✓ ../data/processed/dashboard_data.json generado

=== COMPARACIÓN CÓRDOBA vs LIGA ===
  ppg                                 1.367
  league_ppg                          1.37
  top6_ppg                            1.728
  avg_possession                      57.9
  league_possession                   50.0
  top6_possession                     53.48
  avg_xg                              1.656
  league_xg                           1.351
  top6_xg                             1.502
  avg_xga                             1.423
  league_xga                          1.355
  conversion 

In [11]:
# CELDA 5 — EXPORTACIÓN PARQUETS POWER BI (star schema)

df.to_parquet("../data/warehouse/facts_matches.parquet", index=False)
print(f"✓ facts_matches.parquet  — {len(df)} filas, {len(df.columns)} columnas")

dim_teams = (
    df[["team_id", "team_name"]].drop_duplicates()
    .sort_values("team_name").reset_index(drop=True)
)
dim_teams.to_parquet("../data/warehouse/dim_teams.parquet", index=False)
print(f"✓ dim_teams.parquet      — {len(dim_teams)} equipos")

table.to_parquet("../data/warehouse/dim_standings.parquet", index=False)
print(f"✓ dim_standings.parquet  — {len(table)} equipos")

if "match_date" in df.columns and "round" in df.columns:
    cal_cols = ["match_id", "match_date", "round"]
    if "season_name" in df.columns:
        cal_cols.append("season_name")
    dim_calendar = (
        df[cal_cols].drop_duplicates(subset="match_id")
        .copy().reset_index(drop=True)
    )
    dim_calendar["month"]    = dim_calendar["match_date"].dt.month
    dim_calendar["week"]     = dim_calendar["match_date"].dt.isocalendar().week.astype(int)
    dim_calendar["matchday"] = "J" + dim_calendar["round"].astype(str).str.zfill(2)
    dim_calendar.to_parquet("../data/warehouse/dim_calendar.parquet", index=False)
    print(f"✓ dim_calendar.parquet   — {len(dim_calendar)} partidos")
else:
    print("⚠ dim_calendar no generado: falta match_date o round")

home_rows = df[df["is_home"] == True][["match_id","team_id","team_name","goals_for","result"]].copy()
home_rows.columns = ["match_id","home_team_id","home_team_name","home_goals","result"]
away_rows = df[df["is_home"] == False][["match_id","team_id","team_name","goals_for"]].copy()
away_rows.columns = ["match_id","away_team_id","away_team_name","away_goals"]
dim_matches = home_rows.merge(away_rows, on="match_id", how="inner")
if "match_date" in df.columns:
    dates = df[["match_id","match_date","round"]].drop_duplicates(subset="match_id")
    dim_matches = dim_matches.merge(dates, on="match_id", how="left")
dim_matches.to_parquet("../data/warehouse/dim_matches.parquet", index=False)
print(f"✓ dim_matches.parquet    — {len(dim_matches)} partidos")

perf_by_poss.to_parquet("../data/warehouse/cordoba_possession_performance.parquet", index=False)
print(f"✓ cordoba_possession_performance.parquet — {len(perf_by_poss)} tramos")

print("\n✅ Star schema exportado:")
for f_name in ["facts_matches","dim_teams","dim_standings","dim_calendar","dim_matches","cordoba_possession_performance"]:
    print(f"   {f_name}.parquet")

✓ facts_matches.parquet  — 660 filas, 210 columnas
✓ dim_teams.parquet      — 22 equipos
✓ dim_standings.parquet  — 22 equipos
✓ dim_calendar.parquet   — 330 partidos
✓ dim_matches.parquet    — 330 partidos
✓ cordoba_possession_performance.parquet — 4 tramos

✅ Star schema exportado:
   facts_matches.parquet
   dim_teams.parquet
   dim_standings.parquet
   dim_calendar.parquet
   dim_matches.parquet
   cordoba_possession_performance.parquet


In [12]:
# CELDA 6 — DIAGNÓSTICO Y VERIFICACIÓN FINAL

print("=" * 55)
print("DIAGNÓSTICO FINAL")
print("=" * 55)

# Verificaciones core
check_xg_sum = cor_xg_valid["xg"].sum()
check_conv   = round(float(cor_xg_valid["goals_for"].sum() / check_xg_sum), 4) if check_xg_sum > 0 else None
json_conv    = output["CORDOBA"]["conv_xg"]
conv_ok      = abs((check_conv or 0) - (json_conv or 0)) < 0.01
print(f"conv_xg:   {check_conv} | JSON: {json_conv}  → {'✓' if conv_ok else '✗ MISMATCH'}")

check_xg_mean = round(float(cordoba["xg"].mean()), 2)
json_xg_mean  = output["CORDOBA"]["xg"]
xg_ok = abs(check_xg_mean - (json_xg_mean or 0)) < 0.01
print(f"xG medio:  {check_xg_mean} | JSON: {json_xg_mean}  → {'✓' if xg_ok else '✗ MISMATCH'}")

check_pts = int(cordoba["points"].sum())
pts_ok    = check_pts == output["CORDOBA"]["pts_total"]
print(f"Puntos:    {check_pts} | JSON: {output['CORDOBA']['pts_total']}  → {'✓' if pts_ok else '✗ MISMATCH'}")

# Clasificación
print("\nTop 8 clasificación:")
print(table.head(8)[["position","team_name","points","goal_diff","goals_for","ppg"]].to_string(index=False))

# Cobertura de stats nuevas
print("\nCobertura stats nuevas (% filas con dato):")
check_new = [
    ("shots_in_box",      "Tiros en área"),
    ("shots_out_box",     "Tiros fuera área"),
    ("clearance",         "Despejes"),
    ("errors_lead_shot",  "Errores→tiro"),
    ("dispossessed",      "Pérdidas duelo"),
    ("offsides",          "Fueras de juego"),
    ("free_kicks",        "Faltas recibidas"),
    ("fouled_final_third","Falta último tercio"),
    ("through_balls",     "Pases profundidad"),
    ("hit_woodwork",      "Palos"),
    ("high_claims",       "Salidas aéreas"),
    ("goal_kicks",        "Saques de puerta"),
    ("gk_saves",          "Paradas portero"),
    ("corners",           "Córners"),
    ("fouls",             "Faltas cometidas"),
    ("bigC_scored",       "Grandes oc. conv."),
    ("bigC_missed",       "Grandes oc. fall."),
    ("final_third_entries","Entradas últ. tercio"),
    ("xga",               "xGA"),
    ("xg_net",            "xG neto"),
]
for col_name, label in check_new:
    if col_name in df.columns:
        n_valid = int(df[col_name].notna().sum())
        pct = round(n_valid / len(df) * 100, 1)
        status = "✓" if pct > 80 else ("⚠" if pct > 30 else "✗")
        print(f"  {status} {label:25s}: {pct}%  ({n_valid}/{len(df)})")
    else:
        print(f"  ✗ {label:25s}: columna ausente")

# Top correlaciones
print("\nTop 5 correlaciones con puntos:")
for c in correlaciones_js[:5]:
    print(f"  {c['label']:35s} r={c['r']}")

# NaN del Córdoba
print("\nPartidos Córdoba con NaN:")
check_cor = ["xg","xga","xg_net","poss","total_shots","shotsOT","shots_in_box",
             "touches","tackle","recov","duels","clearance","errors_lead_shot"]
for c in check_cor:
    if c in cordoba.columns:
        n = int(cordoba[c].isna().sum())
        print(f"  {'⚠' if n > 0 else '✓'} {c:20s}: {n if n > 0 else 'completo'}")

print("\n" + "=" * 55)
print("Pipeline completado. Ejecutar update_dashboard.py.")
print("=" * 55)

DIAGNÓSTICO FINAL
conv_xg:   0.7979 | JSON: 0.8  → ✓
xG medio:  1.66 | JSON: 1.66  → ✓
Puntos:    41 | JSON: 41  → ✓

Top 8 clasificación:
 position             team_name  points  goal_diff  goals_for      ppg
        1      Real Racing Club      59         22         62 1.966667
        2               Almería      52         14         56 1.733333
        3   Deportivo La Coruña      52         12         46 1.733333
        4                Málaga      51         12         49 1.700000
        5          CD Castellón      49         12         49 1.633333
        6            Las Palmas      48         15         39 1.600000
        7 Burgos Club de Fútbol      47          6         32 1.566667
        8        Sporting Gijón      45          4         42 1.500000

Cobertura stats nuevas (% filas con dato):
  ✓ Tiros en área            : 100.0%  (660/660)
  ✓ Tiros fuera área         : 100.0%  (660/660)
  ✓ Despejes                 : 100.0%  (660/660)
  ⚠ Errores→tiro             : 